In [ ]:
!pip install sqlalchemy pymysql openpyxl


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
import pandas as pd
import numpy as np
from sqlalchemy import create_engine

# DB mới
username = "local"
password = "dung2006!"
host = "127.0.0.1"
port = "3306"
database = "midterm_proj"

engine = create_engine(
    f"mysql+pymysql://{username}:{password}@{host}:{port}/{database}"
)

print("Connected successfully!")

Connected successfully!


In [ ]:
query = """
SELECT 
    f.firm_id,
    f.ticker,
    ff.fiscal_year AS year,
    ff.current_liabilities,
    ff.total_assets,
    ff.total_liabilities,
    ff.total_equity,
    ff.net_sales,
    ff.net_income,
    ff.operating_income,
    ff.eps_basic,

    fm.shares_outstanding
    fm.share_price,
    fm.market_value_equity,

    fc.net_cfo,
    fc.net_cfi,
    fc.capex,

    fo.state_own,
    fo.foreign_own,

    fi.product_innovation,
    fi.process_innovation,

    meta.employees_count,
    meta.firm_age

FROM fact_financial_year ff
LEFT JOIN dim_firm f 
    ON ff.firm_id = f.firm_id

LEFT JOIN fact_market_year fm
    ON ff.firm_id = fm.firm_id
    AND ff.fiscal_year = fm.fiscal_year
    AND ff.snapshot_id = fm.snapshot_id

LEFT JOIN fact_cashflow_year fc
    ON ff.firm_id = fc.firm_id
    AND ff.fiscal_year = fc.fiscal_year
    AND ff.snapshot_id = fc.snapshot_id

LEFT JOIN fact_ownership_year fo
    ON ff.firm_id = fo.firm_id
    AND ff.fiscal_year = fo.fiscal_year
    AND ff.snapshot_id = fo.snapshot_id

LEFT JOIN fact_innovation_year fi
    ON ff.firm_id = fi.firm_id
    AND ff.fiscal_year = fi.fiscal_year
    AND ff.snapshot_id = fi.snapshot_id

LEFT JOIN fact_firm_year_meta meta
    ON ff.firm_id = meta.firm_id
    AND ff.fiscal_year = meta.fiscal_year
    AND ff.snapshot_id = meta.snapshot_id

WHERE ff.fiscal_year BETWEEN 2019 AND 2024
"""

df = pd.read_sql(query, engine)

print("Total rows:", len(df))
df.head()

Total rows: 100


,firm_id,ticker,year,total_assets,total_liabilities,total_equity,net_sales,net_income,operating_income,eps_basic,...,market_value_equity,net_cfo,net_cfi,capex,state_own,foreign_own,product_innovation,process_innovation,employees_count,firm_age
0,1,AAM,2020,2.108193e+11,1.671828e+10,1.941010e+11,None,-1.196735e+10,None,-1145.0,...,NaN,-2.431145e+10,3.434959e+10,1.370951e+09,0.0,0.0117,1,1,329.0,18.0
1,1,AAM,2021,2.010875e+11,6.761018e+09,1.943265e+11,None,2.255100e+08,None,22.0,...,NaN,4.531249e+10,-2.287699e+09,7.460727e+08,0.0,0.0160,0,1,159.0,19.0
2,1,AAM,2022,2.185795e+11,9.043272e+09,2.095363e+11,None,1.689976e+10,None,1455.0,...,NaN,-3.970805e+09,-4.805325e+10,3.217644e+09,0.0,0.0116,0,1,165.0,20.0
3,1,AAM,2023,2.128573e+11,1.022154e+10,2.026358e+11,None,7.031509e+08,None,40.0,...,NaN,-2.423085e+10,4.400175e+10,2.380000e+08,0.0,0.0103,0,1,176.0,21.0
4,1,AAM,2024,2.023532e+11,6.013144e+09,1.963401e+11,None,-6.295684e+09,None,-602.0,...,NaN,3.404372e+10,5.443430e+09,1.050000e+08,0.0,0.0103,0,1,176.0,22.0


In [ ]:
print("Columns:")
print(df.columns)

print("\nNumber of firms:", df['firm_id'].nunique())
print("Years:", sorted(df['year'].unique()))

Columns:
Index(['firm_id', 'ticker', 'year', 'total_assets', 'total_liabilities',
       'total_equity', 'net_sales', 'net_income', 'operating_income', 'eps_basic',
       'shares_outstanding', 'share_price', 'market_value_equity', 'cfo',
       'cfi', 'capex', 'state_own', 'foreign_own',
       'product_innovation', 'process_innovation', 'employees_count', 'firm_age'],
      dtype='object')

Number of firms: 20
Years: [np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024)]


In [ ]:
# Chuẩn hóa
df.columns = df.columns.str.lower()
df = df.sort_values(['firm_id','year'])

# =============================
# CALCULATED VARIABLES
# =============================

df['calc_debt_ratio'] = df['total_liabilities'] / df['total_assets']
df['calc_roa'] = df['net_income'] / df['total_assets']
df['calc_net_margin'] = df['net_income'] / df['net_sales']

if {'shares_outstanding','share_price'}.issubset(df.columns):
    df['calc_market_cap'] = df['shares_outstanding'] * df['share_price']

# =============================
# QC RULES
# =============================

qc = {}

# COMPLETENESS
qc['missing_core'] = df[['net_sales','total_assets','total_equity']].isnull().any(axis=1)

# RANGE
qc['net_sales_negative'] = df['net_sales'] < 0
qc['assets_negative'] = df['total_assets'] < 0
qc['liabilities_negative'] = df['total_liabilities'] < 0
qc['equity_too_negative'] = df['total_equity'] < -df['total_assets']

if 'ownership_ratio' in df.columns:
    qc['ownership_out_of_range'] = (
        (df['ownership_ratio'] < 0) | (df['ownership_ratio'] > 1)
    )
if 'total_liabilities' in df.columns:
    qc['current_liabilities_negative'] = df['total_liabilities'] < 0
if 'total_share_outstanding' in df.columns:
    qc['total_share_outstanding_invalid'] = df['total_share_outstanding'] <= 0
# FINANCIAL LOGIC
qc['balance_sheet_error'] = (
    abs(df['total_assets'] - (df['total_liabilities'] + df['total_equity']))
    / df['total_assets']
) > 0.03

if 'debt_ratio' in df.columns:
    qc['debt_ratio_error'] = (
        abs(df['debt_ratio'] - df['calc_debt_ratio']) > 0.05
    )

if 'roa' in df.columns:
    qc['roa_error'] = (
        abs(df['roa'] - df['calc_roa']) > 0.05
    )

if 'net_margin' in df.columns:
    qc['net_margin_error'] = (
        abs(df['net_margin'] - df['calc_net_margin']) > 0.05
    )

if 'market_value_equity' in df.columns:
    qc['market_cap_error'] = (
        abs(df['market_value_equity'] - df['calc_market_cap'])
        / df['market_value_equity'] > 0.05
    )

# TIME SERIES
df['net_sales_growth'] = df.groupby('firm_id')['net_sales'].pct_change()
qc['net_sales_growth_abnormal'] = (
    (df['net_sales_growth'] > 5) | (df['net_sales_growth'] < -0.95)
)

df['assets_growth'] = df.groupby('firm_id')['total_assets'].pct_change()
qc['assets_growth_abnormal'] = abs(df['assets_growth']) > 3
# LOGICAL CONSISTENCY CHECKS
tolerance = 0.01  # 1% sai số cho phép

if all(col in df.columns for col in ['total_assets','total_liabilities','total_equity']):
    qc['accounting_mismatch'] = (
        abs(df['total_assets'] - (df['total_liabilities'] + df['total_equity']))
        > tolerance * df['total_assets']
    )

if all(col in df.columns for col in ['market_value_equity',
                                     'total_share_outstanding',
                                     'share_price']):
    qc['market_cap_mismatch'] = (
        abs(
            df['market_value_equity']
            - df['total_share_outstanding'] * df['share_price']
        )
        > 0.02 * df['market_value_equity']
    )

#CROSS-VATIABLE LOGIC CHECKS 
if all(col in df.columns for col in ['net_sales','net_income']):
    qc['income_without_net_sales'] = (
        (df['net_sales'] == 0) & (df['net_income'] != 0)
    )

if all(col in df.columns for col in ['innovation_dummy','rd_expense']):
    qc['innovation_without_rd'] = (
        (df['innovation_dummy'] == 1) & (df['rd_expense'] <= 0)
    )


# GROWTH AND OUTLIER CHECKS
if 'net_sales_growth' in df.columns:
    qc['extreme_growth'] = (
        (df['net_sales_growth'] > 10) | (df['net_sales_growth'] < -0.99)
    )

df = df.sort_values(['ticker','fiscal_year'])
df['rev_lag'] = df.groupby('ticker')['net_sales'].shift(1)

qc['net_sales_jump'] = (
    abs((df['net_sales'] - df['rev_lag']) / df['rev_lag']) > 5
)

# STRUCTUTAL/ DATA COMPLETENESS CHECKS 
core_cols = ['net_sales','total_assets','net_income']

for col in core_cols:
    qc[f'missing_{col}'] = df[col].isna()

if all(col in df.columns for col in ['capex','cashflow_investing']):
    qc['capex_inconsistency'] = (
        (df['capex'] > 0) & (df['cashflow_investing'] > 0)
    )

# FINANCIAL RATIO SANITY CHECKS 
if all(col in df.columns for col in ['net_income','total_assets']):
    roa = df['net_income'] / df['total_assets']
    qc['extreme_roa'] = abs(roa) > 2

if all(col in df.columns for col in ['total_liabilities','total_assets']):
    debt_ratio = df['total_liabilities'] / df['total_assets']
    qc['debt_ratio_above_1'] = debt_ratio > 1.5
# =============================
# CREATE QC TABLE
# =============================

qc_df = pd.DataFrame(qc)
total_obs = len(df)

results = []

for rule in qc_df.columns:
    errors = qc_df[rule].sum()
    accuracy = (total_obs - errors) / total_obs * 100
    
    results.append({
        'rule': rule,
        'errors': errors,
        'error_rate_%': round(errors/total_obs*100,2),
        'accuracy_%': round(accuracy,2)
    })

summary = pd.DataFrame(results)

summary

C:\Users\minhd\AppData\Local\Temp\ipykernel_6280\734118064.py:64: FutureWarning: The default fill_method='ffill' in SeriesGroupBy.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  df['net_sales_growth'] = df.groupby('firm_id')['net_sales'].pct_change()
C:\Users\minhd\AppData\Local\Temp\ipykernel_6280\734118064.py:69: FutureWarning: The default fill_method='ffill' in SeriesGroupBy.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  df['assets_growth'] = df.groupby('firm_id')['total_assets'].pct_change()


,rule,errors,error_rate_%,accuracy_%
0,missing_core,100,100.0,0.0
1,net_sales_negative,0,0.0,100.0
2,assets_negative,0,0.0,100.0
3,liabilities_negative,0,0.0,100.0
4,equity_too_negative,0,0.0,100.0
5,balance_sheet_error,0,0.0,100.0
6,market_cap_error,0,0.0,100.0
7,net_sales_growth_abnormal,0,0.0,100.0
8,assets_growth_abnormal,0,0.0,100.0


In [ ]:
total_errors = qc_df.sum().sum()
total_possible = total_obs * len(qc_df.columns)

overall_accuracy = (total_possible - total_errors) / total_possible * 100

print("Total observations:", total_obs)
print("Total rules:", len(qc_df.columns))
print("Overall Accuracy:", round(overall_accuracy,2), "%")

Total observations: 100
Total rules: 9
Overall Accuracy: 88.89 %


In [ ]:
firm_summary = []

for firm, group in df.groupby('firm_id'):
    sub_qc = qc_df.loc[group.index]
    total = len(group) * len(qc_df.columns)
    errors = sub_qc.sum().sum()
    acc = (total - errors) / total * 100
    
    firm_summary.append({
        'firm_id': firm,
        'accuracy_%': round(acc,2)
    })

pd.DataFrame(firm_summary)

,firm_id,accuracy_%
0,1,88.89
1,2,88.89
2,3,88.89
3,4,88.89
4,5,88.89
5,6,88.89
6,7,88.89
7,8,88.89
8,9,88.89
9,10,88.89
